In [2]:
import torch
from einops import rearrange
from llm.nn import scaled_dot_product_attention

In [3]:
# h, num heads
# d_model, embedding dim
# d_k = d_v = d_model / h
# Q.shape -> (... seq_len h*d_k)
# K.shape -> (... seq_len h*d_k)
# V.shape -> (... seq_len h*d_v)
# Wq.shape -> h*d_k, d_model
# Wk.shape -> h*d_k, d_model
# Wv.shape -> h*d_v, d_model
# Wo.shape -> d_model, h*d_v
# MultiHead(𝑄, 𝐾, 𝑉 ) = Concat(head_1 , …, head_h )
#     for head_𝑖 = Attention(𝑄𝑖 , 𝐾𝑖 , 𝑉𝑖 )
# MultiHeadSelfAttention(𝑥) = Wo MultiHead(𝑊𝑄 𝑥, 𝑊𝐾 𝑥, 𝑊𝑉 𝑥)
# Qi.shape -> (... seq_len d_k) 
# x.shape -> (... seq_len d_model)

In [46]:
batch_size = 1
seq_len = 5
d_model = 32
h = 2
d_k = d_v = d_model // h

In [47]:
x = torch.randn(batch_size, seq_len, d_model)
# print(x)
Wq = torch.randn(1, h*d_k, d_model)
Wk = torch.randn(1, h*d_k, d_model)
Wv = torch.randn(1, h*d_v, d_model)
Wo = torch.randn(batch_size, d_model, d_model)

Wo_transpose = rearrange(Wq, "... dm1 dm2 -> ... dm2 dm1")
Wq_transpose = rearrange(Wq, "... dm1 dm2 -> ... dm2 dm1")
Wk_transpose = rearrange(Wk, "... dm1 dm2 -> ... dm2 dm1")
Wv_transpose = rearrange(Wv, "... dm1 dm2 -> ... dm2 dm1")

print(Wq_transpose.shape, x.shape)

seq_len = x.shape[-2]
print(f"{seq_len = }")

torch.Size([1, 32, 32]) torch.Size([1, 5, 32])
seq_len = 5


In [48]:
Q = x @ Wq_transpose 
K = x @ Wk_transpose 
V = x @ Wv_transpose

Q = rearrange(Q, 'batch_size seq_len (d_k num_heads) -> batch_size num_heads seq_len d_k', num_heads=h)
K = rearrange(K, 'batch_size seq_len (d_k num_heads) -> batch_size num_heads seq_len d_k', num_heads=h)
V = rearrange(V, 'batch_size seq_len (d_k num_heads) -> batch_size num_heads seq_len d_k', num_heads=h)

print(f"{Q.shape = }")
print(f"{K.shape = }")
print(f"{V.shape = }")

Q.shape = torch.Size([1, 2, 5, 16])
K.shape = torch.Size([1, 2, 5, 16])
V.shape = torch.Size([1, 2, 5, 16])


(torch.Size([1, 32, 32]),
 tensor([[[ 1.6002,  0.3778, -0.7565,  ...,  1.0256,  0.1986,  0.0276],
          [-0.0508, -0.1280, -0.7841,  ...,  0.6295, -0.2866, -1.3185],
          [-0.1269, -0.1025,  0.5502,  ..., -0.7776,  0.1640, -2.3415],
          ...,
          [-0.3701, -1.0904,  2.7732,  ..., -0.3331,  0.7975, -0.0272],
          [ 0.5113,  0.0672,  0.9621,  ..., -0.7106, -0.2551,  0.0509],
          [-0.2794, -0.6931, -0.5697,  ..., -0.6694,  0.6291,  0.7797]]]))

In [53]:
mask = ~torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), 1)

In [54]:
attn = rearrange(attn, 'batch num_heads seq_len d_k -> batch seq_len (num_heads d_k)')
attn.shape

torch.Size([1, 5, 10])

In [55]:
Wo = rearrange(Wo, 'b d_m d_m2 -> b d_m2 d_m')
Wo

tensor([[[ 2.8731, -0.6016,  0.0686,  ..., -0.3437, -0.1614, -0.2076],
         [-0.2044, -1.6726,  0.0876,  ...,  0.2681,  2.0010,  0.0646],
         [ 0.0599,  0.1733, -0.2630,  ..., -0.2682, -1.3090, -0.4205],
         ...,
         [-0.7867, -0.1090, -1.2916,  ...,  0.4209,  1.0059,  1.0271],
         [-0.2254,  1.6806, -0.3036,  ...,  0.0038, -1.8427,  0.2379],
         [ 0.4731,  0.5912, -0.7765,  ..., -0.2601, -1.1261, -0.5437]]])

In [56]:
out = attn @ Wo
out, out.shape

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [1, 10] but got: [1, 32].

In [ ]:
def causal_multihead_attention(x: torch.Tensor, d_model: int, num_heads: int) -> torch.Tensor:
    pass
    
def multi_head(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, num_heads: int, mask: torch.Tensor | None = None) -> torch.Tensor:
    pass

